In [1]:
from bertopic import BERTopic
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def bertopic_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    bertopic_analysis(nurse_notes[key])
    all_texts.extend(nurse_notes[key])

-----------P1-----------
Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6711261642870752
Diversity: 0.49411764705882355
Inverse Redundancy: 0.8713235294117647
Time (seconds): 6.415049076080322
----- Cluster Topics -----
['resident', 'give', 'doctor', 'injection', 'need', 'attend', 'night', 'concern', 'med', 'voice']
['check', 'safety', 'night', 'comfortable', 'take', 'continue', 'med', 'sleep', 'concern', 'chart']
['sleep', 'self', 'remain', 'asleep', 'pleasant', 'ongoing', 'check', 'care', 'hourly', 'pleasantly']
['walker', 'mobilizing', 'relaxed', 'take', 'staff', 'chart', 'med', 'content', 'appear', 'assist']
['plan', 'morning', 'adls', 'breakfast', 'report', 'staff', 'intake', 'room', 'interact', 'care']
['restaurant', 'mobile', 'attend', 'independent', 'usual', 'need', 'take', 'meal', 'form', 'med']
['toilette', 'ongoing', 'asleep', 'comfortable', 'self', 'check', 'resident', 'toilete', 'peacefully', 'require']
['peaceful', 'toilette', 'ongoing', 'asleep', 'self', 'check', 'resident', 'toilete', 'antibiotic', 'safety']
['steroid', 'good',

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7421585978838229
Diversity: 0.675
Inverse Redundancy: 0.95
Time (seconds): 13.013169050216675
----- Cluster Topics -----
['care', 'chair', 'plan', 'resident', 'good', 'chart', 'assist', 'day', 'med', 'form']
['settle', 'tts', 'comfortably', 'hoist', 'night', 'staff', 'medication', 'give', 'bed', 'drink']
['check', 'night', 'safety', 'comfortable', 'continue', 'concern', 'sleep', 'need', 'resident', 'care']
['laxative', 'bno', 'microlax', 'give', 'prn', 'morning', 'remain', 'refuse', 'oral', 'content']
['vomiting', 'nausea', 'vomit', 'episode', 'feel', 'report', 'urine', 'bell', 'later', 'watch']
['wheelchair', 'baseline', 'electric', 'prescribe', 'wash', 'transfer', 'skin', 'take', 'form', 'appear']
['adls', 'adequate', 'new', 'intake', 'nil', 'pu', 'chart', 'appear', 'concern', 'give']
['pressure', 'sit', 'comfort', 'bed', 'television', 'place', 'bunion', 'bell', 'later', 'area']
['receive', 'till', 'ensure', 'time', 'room', 'new', 'note', 'keep', 'med', 'issue']
['eye', 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7665271401869582
Diversity: 0.5263157894736842
Inverse Redundancy: 0.9029239766081871
Time (seconds): 2.7920339107513428
----- Cluster Topics -----
['resident', 'bed', 'care', 'form', 'good', 'daughter', 'med', 'morning', 'give', 'assist']
['adls', 'assisted', 'meds', 'charted', 'compliant', 'maintain', 'night', 'have', 'settle', 'safety']
['eye', 'attend', 'form', 'good', 'care', 'voice', 'complaint', 'medication', 'appear', 'chart']
['adls', 'charted', 'compliant', 'assisted', 'meds', 'maintain', 'settle', 'night', 'safety', 'need']
['medication', 'administer', 'alert', 'bright', 'appear', 'new', 'house', 'eating', 'drink', 'concern']
['entry', 'time', 'personal', 'potter', 'concern', 'voice', 'good', 'room', 'bright', 'give']
['asleep', 'mat', 'place', 'comfortable', 'sensor', 'check', 'issue', 'med', 'bed', 'appear']
['toilet', 'comfortable', 'nocte', 'early', 'asleep', 'bed', 'place', 'check', 'mat', 'sensor']
['toilete', 'gong', 'place', 'mat', 'sensor', 'assist', 's

In [7]:
bertopic_analysis(all_texts)

Number of texts: 12225


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.5921785060833745
Diversity: 0.3062314540059347
Inverse Redundancy: 0.9662091988130563
Time (seconds): 18.074954986572266
----- Cluster Topics -----
['gp', 'note', 'doctor', 'receive', 'pain', 'sleep', 'till', 'keep', 'med', 'morning']
['inhaler', 'club', 'social', 'nebs', 'aspiration', 'antibiotic', 'laxose', 'listen', 'music', 'knitting']
['wheel', 'observe', 'intake', 'today', 'chair', 'pu', 'note', 'aid', 'skin', 'medicine']
['complaint', 'voice', 'slt', 'medication', 'attend', 'adhere', 'rolator', 'nil', 'take', 'activity']
['alert', 'medicine', 'food', 'bright', 'fluid', 'render', 'intake', 'today', 'intact', 'pu']
['actively', 'integrity', 'general', 'supervised', 'main', 'progress', 'insitu', 'post', 'randomly', 'toilet']
['compliant', 'charted', 'meds', 'assisted', 'adls', 'maintain', 'night', 'settle', 'safety', 'versatili']
['diet', 'weight', 'nutritional', 'kg', 'dietetic', 'dietitian', 'protein', 'loss', 'vitamin', 'year']
['mobile', 'restaurant', 'independent'